In [0]:
-- ============================================================================
-- TASK 3: ATOMIC SWAP + CLEANUP
-- ============================================================================
-- PURPOSE  : Promotes the validated staging table to live using a 2-step
--            DROP + RENAME pattern. No backup is kept — validation in Task 2
--            is the sole quality gate before the swap.
--
-- STEPS    :
--   1. Drop the current live table (frees the name for the rename)
--   2. Rename staging → live (promotes staging to live instantly)
--   3. Stamp table properties with refresh metadata
--   4. OPTIMIZE + ZORDER for downstream query performance
--
-- NOTE     : RENAME in Delta Lake is metadata-only (near-instant).
--            The view (fin_metric_growth_vw) ensures consumers never see
--            a "table not found" error during the swap window.
--
-- DEPENDS ON : task2_validate_staging.sql (set as dependency in Workflow)
-- ============================================================================


-- ----------------------------------------------------------------------------
-- STEP 1: Drop the current live table to free the name for RENAME
-- Safe to drop because:
--   a) Consumers query the view, not this table directly
--   b) Task 2 validation has already confirmed staging is good
-- ----------------------------------------------------------------------------
DROP TABLE IF EXISTS
  furlenco_analytics.materialized_tables.fin_metric_query_growth;


-- ----------------------------------------------------------------------------
-- STEP 2: Promote staging to live
-- RENAME is metadata-only in Delta — completes in milliseconds
-- ----------------------------------------------------------------------------
ALTER TABLE furlenco_analytics.materialized_tables.fin_metric_query_growth_staging
  RENAME TO furlenco_analytics.materialized_tables.fin_metric_query_growth;


-- ----------------------------------------------------------------------------
-- STEP 3: Stamp refresh metadata on the live table
-- EXECUTE IMMEDIATE is required in Databricks because SET TBLPROPERTIES does
-- not accept function calls like current_timestamp() directly — the value
-- must be evaluated and injected as a string via dynamic SQL.
-- Check freshness anytime with:
--   SHOW TBLPROPERTIES furlenco_analytics.materialized_tables.fin_metric_query_growth;
-- ----------------------------------------------------------------------------
EXECUTE IMMEDIATE
  CONCAT(
    "ALTER TABLE furlenco_analytics.materialized_tables.fin_metric_query_growth SET TBLPROPERTIES (",
    "'last_refreshed_at' = '", string(current_timestamp()), "', ",
    "'refresh_type' = 'full_refresh', ",
    "'refresh_schedule' = 'every_4_hours')"
  );


-- ----------------------------------------------------------------------------
-- STEP 4: Optimize the live table for downstream query performance
-- ZORDER co-locates data by the most commonly filtered/joined columns,
-- speeding up queries that filter on these dimensions.
-- Columns chosen based on most frequent WHERE/GROUP BY patterns for
-- growth team dashboards (state, vertical, city, date, order type).
-- ----------------------------------------------------------------------------
OPTIMIZE furlenco_analytics.materialized_tables.fin_metric_query_growth
  ZORDER BY (order_state, current_entity_state, vertical, product_entity_type, order_type, city, transaction_date);


-- ----------------------------------------------------------------------------
-- Final confirmation — output appears in the Workflow run log
-- ----------------------------------------------------------------------------
SELECT
  'SWAP SUCCESSFUL'       AS status,
  COUNT(*)                AS live_row_count,
  MIN(transaction_date)   AS earliest_transaction,
  MAX(transaction_date)   AS latest_transaction,
  current_timestamp() + interval '330 minutes'    AS swapped_at
FROM furlenco_analytics.materialized_tables.fin_metric_query_growth;